# Part 1 - Multimodal RAG
DATA 266 Lab 2, Pair_Programming_Team_32. The code lives in `part1_rag/src/`; this notebook runs it.
Runs on **Kaggle** (the graded path), Colab, or locally. CPU is enough: Restart & Run All.


In [ ]:
# Cell 1 - run information (required)
import subprocess, sys, random, numpy as np
SEED = 266; random.seed(SEED); np.random.seed(SEED)
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
try:
    import torch; gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
except ImportError:
    gpu = "cpu"
print({"commit": commit, "python": sys.version.split()[0], "gpu": gpu, "seed": SEED})

## Setup
- **Kaggle:**
  - Settings: Internet **on**.
  - Add-ons → Secrets: `ANTHROPIC_API_KEY`. `GEMINI_API_KEY` is optional; it's only needed if a figure is missing from the committed transcription cache.
  - Add data:
    - the competition, which supplies the PDFs, `Questions.csv`, `kb_manifest.json` and the validator;
    - a private Kaggle dataset holding `repo.bundle` (made with `git bundle create repo.bundle zoheb`).
- **Colab:** secrets `ANTHROPIC_API_KEY` and `GEMINI_API_KEY`. Upload `repo.bundle` when asked.


In [ ]:
import os, glob
BRANCH = "zoheb"                                            # switch to main once merged
ON_KAGGLE, ON_COLAB = os.path.isdir("/kaggle/input"), "google.colab" in sys.modules
if ON_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ["ANTHROPIC_API_KEY"] = secrets.get_secret("ANTHROPIC_API_KEY")
    try:
        os.environ["GEMINI_API_KEY"] = secrets.get_secret("GEMINI_API_KEY")
    except Exception:
        pass                                                # only needed on a figure-cache miss
    # the competition files are read-only under /kaggle/input; the code comes from the attached repo.bundle dataset
    os.environ["P1_DATA"] = os.path.dirname(glob.glob("/kaggle/input/**/Questions.csv", recursive=True)[0])
    bundle = glob.glob("/kaggle/input/**/repo.bundle", recursive=True)[0]
    os.chdir("/kaggle/working")
    if not os.path.isdir("data266-lab2-team32"):
        !git clone -q -b {BRANCH} {bundle} data266-lab2-team32
    %cd /kaggle/working/data266-lab2-team32
    !pip install -q pymupdf chromadb sentence-transformers rank-bm25 google-genai anthropic pydantic
elif ON_COLAB:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    if not os.path.isdir("data266-lab2-team32"):
        from google.colab import files                      # no GitHub token: upload repo.bundle (git bundle create repo.bundle zoheb)
        files.upload()
        !git clone -q -b {BRANCH} repo.bundle data266-lab2-team32
    %cd data266-lab2-team32
    !pip install -q pymupdf chromadb sentence-transformers rank-bm25 google-genai anthropic pydantic
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
import anthropic, chromadb, pymupdf, sentence_transformers, google.genai
from part1_rag.src import common
print({"commit": subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip(),
       "data": str(common.DATA), "pymupdf": pymupdf.__version__, "chromadb": chromadb.__version__,
       "sentence-transformers": sentence_transformers.__version__, "google-genai": google.genai.__version__, "anthropic": anthropic.__version__,
       "answers": common.CLAUDE_MODEL, "figure transcriptions": common.GEMINI_MODEL})


## Data
The competition files are not in git. Kaggle reads them from the attached competition data. On Colab, upload the 3 PDFs and `Questions.csv` when asked.


In [ ]:
need = [d["file"] for d in common.DOCS.values()] + ["Questions.csv"]
missing = [f for f in need if not (common.DATA / f).exists()]
if missing and "google.colab" in sys.modules:
    from google.colab import files
    print("upload:", missing)
    for name, data in files.upload().items():
        (common.DATA / name.replace(" (1)", "")).write_bytes(data)
missing = [f for f in need if not (common.DATA / f).exists()]
assert not missing, f"missing data files: {missing}"

## 1.1 Ingest and index
Pages become text chunks; every figure and table is cropped. Figure transcriptions come from the committed
cache `outputs/figure_descriptions.json`, so this step makes no API calls.

In [ ]:
from part1_rag.src import ingest, index
chunks, figures = ingest.ingest_all()
index.build_index()

## 1.2-1.5 Answer Questions.csv
One Claude call per question (40 for the public set). Paid API: roughly $1-2 per full run on Sonnet 5.5.

In [ ]:
from part1_rag.src import pipeline
sub, seconds = pipeline.run()
print("median seconds/question:", round(float(np.median(list(seconds.values()))), 2))
print(sub["status"].value_counts().to_dict())
assert pipeline.validate()
sub.head(10)

In [ ]:
import shutil
if ON_KAGGLE:                                               # Kaggle takes the submission from /kaggle/working
    shutil.copy(common.OUTPUTS / "submission.csv", "/kaggle/working/submission.csv")
    print("wrote /kaggle/working/submission.csv")
elif ON_COLAB:
    files.download(str(common.OUTPUTS / "submission.csv"))
